<a href="https://colab.research.google.com/github/Nastya2006Fed/python-ai-Nastya-Fedorova/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- **[`rivers.csv`](https://github.com/Nastya2006Fed/python-ai-Nastya-Fedorova/blob/main/data/rivers.csv)** — информация о реках: название, страна, длина, исток и устье

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [1]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

# 🔁 ИЗМЕНЕНО: имя репозитория и URL
repo = "python-ai-Nastya-Fedorova"  # было: "python-ai-template"
repo_path = f"/content/{repo}"

if not os.path.exists(repo_path):
    # 🔁 ИЗМЕНЕНО: URL вашего репозитория
    !git clone -q https://github.com/Nastya2006Fed/python-ai-Nastya-Fedorova.git

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Nastya-Fedorova
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Nastya-Fedorova


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [10]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd

df_rivers = pd.read_csv("data/rivers.csv")

print("✅ Загружено строк в df_genre:", len(df_rivers))


✅ Загружено строк в df_genre: 563


## 🧹 [2B] Очистка и переименование столбцов

В вашем CSV-файле `rivers.csv` есть **технические столбцы**, которые полезны для Викиданных, но мешают простому анализу:

- Столбец `river` с URL (ссылкой на объект Wikidata) — **удаляем**, так как для базового анализа он не требуется.
- Столбцы `riverLabel`, `countryLabel`, `sourceLabel`, `mouthLabel` содержат читаемые подписи (названия) — **убираем суффикс `Label`** для удобства.

В этом шаге мы:
- 🔁 **удаляем столбец `river` с URL Wikidata**;
- 🔁 **переименовываем**: `riverLabel → river`, `countryLabel → country`, `sourceLabel → source`, `mouthLabel → mouth`;
- 🔁 **приводим столбец `length` к числовому типу** (`int`).

При приведении к числам мы используем:
- `pd.to_numeric(..., errors="coerce")` — преобразует значения в числа, некорректные значения превращает в `NaN`;
- `fillna(0)` — заменяет пропущенные значения (`NaN`) на 0;
- `astype(int)` — переводит столбец к целочисленному типу.

> ⚠️ **Важно:** если в ваших данных есть столбцы с URL Wikidata и столбцы вида `*Label`, этот шаг обязателен, чтобы получить аккуратные таблички для анализа. Столбец с URL можно сохранить, если нужно будет быстро перейти к оригинальной записи в Викиданных.

In [11]:
# 🧹 Шаг 2B. Очистка и переименование столбцов для rivers.csv

from IPython.display import display  # 🔁 ДОБАВЛЕНО: для красивого вывода таблиц

if "riverLabel" in df_rivers.columns:                             # Проверка: нужна ли очистка?

    # Удаляем столбец с URL Wikidata (не нужен для анализа)
    if "river" in df_rivers.columns and df_rivers["river"].dtype == "object" and df_rivers["river"].str.contains("wikidata", case=False, na=False).any():
        df_rivers = df_rivers.drop(columns=["river"])

    # Переименовываем столбцы, убирая суффикс Label
    df_rivers = df_rivers.rename(columns={
        "riverLabel": "river",          # ← название реки
        "countryLabel": "country",      # ← страна
        "sourceLabel": "source",        # ← исток
        "mouthLabel": "mouth",          # ← устье
    })

    # Приводим длину реки к числовому типу
    df_rivers["length"] = pd.to_numeric(
        df_rivers["length"], errors="coerce"
    ).fillna(0).astype(int)

    print("✅ df_rivers очищен и переименован")
else:
    print("⏭️ df_rivers уже очищен, пропускаем")

# 🔁 ИСПРАВЛЕНО: вывод таблицы без print() для HTML-форматирования
print("\n📋 Столбцы после очистки:", df_rivers.columns.tolist())
display(df_rivers.head())  # ✅ Красивая таблица вместо текста

print("\n✅ Данные готовы к анализу")

✅ df_rivers очищен и переименован

📋 Столбцы после очистки: ['river', 'country', 'length', 'source', 'mouth']


,river,country,length,source,mouth
0,Снейк,США,1735,Йеллоустонский национальный парк,Колумбия
1,Куараи,Уругвай,351,Бразилия,Уругвай
2,Куараи,Бразилия,351,Бразилия,Уругвай
3,Сож,Россия,648,Россия,Днепр
4,Куараи,Уругвай,351,Бразилия,Уругвай



✅ Данные готовы к анализу


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор обоих DataFrame:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по бюджету (`capital_cost`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы не повторять один и тот же код два раза.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

# 🔍 Шаг 3. Обзор данных

show_info(df_genre, "Жанры, страны и продолжительность (df_genre)")
show_info(df_assessment, "Год выпуска, оценки и рейтинги (df_assessment)")


📊 Жанры, страны и продолжительность (df_genre)
Размер: (2596, 6)
Столбцы: URL, film, genre, country, duration, capital_cost

Первые строки:
                                       URL                       film  \
0  http://www.wikidata.org/entity/Q1128756                Мэри и Макс   
1  http://www.wikidata.org/entity/Q1199692  Отважный маленький тостер   
2  http://www.wikidata.org/entity/Q1514402                  Ренессанс   
3  http://www.wikidata.org/entity/Q1514402                  Ренессанс   
4  http://www.wikidata.org/entity/Q1418615     Приключения кота Фрица   

                                    genre     country  duration  capital_cost  
0                       взрослая анимация   Австралия        90     8240000.0  
1  экранизация литературного произведения         США        90     2300000.0  
2                               киберпанк  Люксембург       101    18000000.0  
3                                 неонуар     Франция       101    18000000.0  
4  экранизация литер

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** фильмов, стран, жанров есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу строк);
- **какие жанры самые популярные** (Топ‑10 по числу строк);
- **какие оценки (assessment) и результаты (outcome) присутствуют** в данных об оценках.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_genre["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет 1: жанры, страны, длительность
print("\n📊 Датасет: Жанры, страны и продолжительность")
print("Уникальных мультфильмов в df_genre:", df_genre["film"].nunique())
print("Уникальных стран:", df_genre["country"].nunique())
print("Уникальных жанров:", df_genre["genre"].nunique())

print("\nТоп-5 стран по числу записей:")
print(df_genre["country"].value_counts().head())

print("\nТоп-10 жанров:")
print(df_genre["genre"].value_counts().head(10))

# Датасет 2: оценки и рейтинги
print("\n📊 Датасет: Год выпуска, оценки и рейтинги")
print("Уникальных мультфильмов в df_assessment:", df_assessment["film"].nunique())
print("Уникальных типов оценок:", df_assessment["assessment"].nunique())
print("Диапазон лет:", df_assessment["publicationYear"].min(), "—", df_assessment["publicationYear"].max())

print("\nТипы оценок (assessment):")
print(df_assessment["assessment"].value_counts())

print("\nРезультаты оценок (outcome):")
print(df_assessment["outcome"].value_counts())

print("\nСтатистика по годам публикации:")
print(df_assessment["publicationYear"].describe())

🔍 Быстрая проверка данных

📊 Датасет: Жанры, страны и продолжительность
Уникальных мультфильмов в df_genre: 418
Уникальных стран: 40
Уникальных жанров: 125

Топ-5 стран по числу записей:
country
США               1303
Франция            196
Великобритания     125
Россия              91
Дания               82
Name: count, dtype: int64

Топ-10 жанров:
genre
приключенческий фильм          353
комедийный фильм               305
фэнтезийный фильм              289
семейный фильм                 247
детский фильм                  187
музыкальный фильм              120
драматический фильм             91
боевик                          84
научно-фантастический фильм     74
мультфильм                      70
Name: count, dtype: int64

📊 Датасет: Год выпуска, оценки и рейтинги
Уникальных мультфильмов в df_assessment: 4929
Уникальных типов оценок: 22
Диапазон лет: 0 — 2027

Типы оценок (assessment):
assessment
тест Бекдел                                                                   1426
обрат

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали 2 CSV-файла из `data/examples/`
- ✅ Удалили URL Wikidata и переименовали столбцы (`*Label → короткие имена`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных фильмов, стран, жанров
  - диапазоны значений
  - топ стран и жанров по числу записей
  - типы оценок и результатов

Теперь у нас есть **аккуратные, проверенные таблицы**, с которыми удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки, фильтрация),
- и построения визуализаций (графики и диаграммы). 🎨